# EDA - Sparq Challenge
## UK Road Safety Data
### Miriam Eunice Rosas Medellín
#### Data Exploration using SQL

In [ ]:
SHOW TABLES IN SCHEMA SPARQ_CHALLENGE.BRONZE;

In [ ]:
SHOW TABLES IN SCHEMA SPARQ_CHALLENGE.SILVER;

In [ ]:
SELECT 
    table_name,
    column_name,
    data_type,
    is_nullable
FROM SPARQ_CHALLENGE.INFORMATION_SCHEMA.COLUMNS
WHERE table_schema = 'BRONZE' OR table_schema = 'SILVER'
ORDER BY table_name, ordinal_position;

In [ ]:
SELECT table_name, row_count
FROM SPARQ_CHALLENGE.INFORMATION_SCHEMA.TABLES
WHERE table_schema = 'BRONZE'
  AND table_type = 'BASE TABLE'
ORDER BY table_name;

In [ ]:
SELECT table_name, row_count
FROM SPARQ_CHALLENGE.INFORMATION_SCHEMA.TABLES
WHERE table_schema = 'SILVER'
  AND table_type = 'BASE TABLE'
ORDER BY table_name;

#### EDA using Python
Structure:
- Phase 1 — Setup                        (Cells 1–3)
- Phase 2 — Univariate quality           (Cells 4–8)
- Phase 3 — Referential integrity        (Cells 9–14)
- Phase 4 — Exploratory / business       (Cells 15–20)
- Phase 5 — Summary                      (Cell 21)

##### Phase 1: Setup

In [ ]:
# ---------------------------------------------------------------------------
# Cell 1 — Imports & session setup
# ---------------------------------------------------------------------------
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from snowflake.snowpark.context import get_active_session
 
# Initialize Snowpark session (already available in Snowflake Notebooks)
session = get_active_session()
 
# Global plot style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)
 
print("Session active:", session.get_current_database(), "|", session.get_current_schema())

In [ ]:
# ---------------------------------------------------------------------------
# Cell 2 — Load all fact tables and dimension tables
# ---------------------------------------------------------------------------
 
# --- Fact tables (Bronze) ---
accidents = session.table("SPARQ_CHALLENGE.BRONZE.ACCIDENTS").to_pandas()
accidents_2019 = session.table(
    "SPARQ_CHALLENGE.BRONZE.ACCIDENTS_2019_MIDYEAR_PROVISIONAL"
).to_pandas()
casualties = session.table("SPARQ_CHALLENGE.BRONZE.CASUALTIES").to_pandas()
vehicles = session.table("SPARQ_CHALLENGE.BRONZE.VEHICLES").to_pandas()
blood_alcohol = session.table("SPARQ_CHALLENGE.BRONZE.BLOOD_ALCOHOL_CONTENT").to_pandas()
 
# Standardize column names: strip whitespace, uppercase, replace special chars
all_fact_dfs = {
    "ACCIDENTS":      accidents,
    "ACCIDENTS_2019": accidents_2019,
    "CASUALTIES":     casualties,
    "VEHICLES":       vehicles,
    "BLOOD_ALCOHOL":  blood_alcohol,
}
for df in all_fact_dfs.values():
    df.columns = (
        df.columns
        .str.strip()
        .str.upper()
        .str.replace(r"[\s\(\)\-\?]", "_", regex=True)
    )
 
# --- Dimension tables (Silver) ---
# Loaded once here and reused across all phases to avoid redundant queries
dim_sources = {
    "ACCIDENT_SEVERITY":   "SPARQ_CHALLENGE.SILVER.DIM_ACCIDENT_SEVERITY",
    "SPEED_LIMIT":         "SPARQ_CHALLENGE.SILVER.DIM_SPEED_LIMIT",
    "URBAN_OR_RURAL_AREA": "SPARQ_CHALLENGE.SILVER.DIM_URBAN_RURAL",
    "LIGHT_CONDITIONS":    "SPARQ_CHALLENGE.SILVER.DIM_LIGHT_CONDITIONS",
    "WEATHER_CONDITIONS":  "SPARQ_CHALLENGE.SILVER.DIM_WEATHER",
    "ROAD_TYPE":           "SPARQ_CHALLENGE.SILVER.DIM_ROAD_TYPE",
    "ROAD_SURFACE_CONDITIONS": "SPARQ_CHALLENGE.SILVER.DIM_ROAD_SURFACE",
    "JUNCTION_DETAIL":     "SPARQ_CHALLENGE.SILVER.DIM_JUNCTION_DETAIL",
    "JUNCTION_CONTROL":    "SPARQ_CHALLENGE.SILVER.DIM_JUNCTION_CONTROL",
    "CARRIAGEWAY_HAZARDS": "SPARQ_CHALLENGE.SILVER.DIM_CARRIAGEWAY_HAZARDS",
    "VEHICLE_TYPE":        "SPARQ_CHALLENGE.SILVER.DIM_VEHICLE_TYPE",
    "VEHICLE_MANOEUVRE":   "SPARQ_CHALLENGE.SILVER.DIM_VEHICLE_MANOEUVRE",
    "SKIDDING_AND_OVERTURNING": "SPARQ_CHALLENGE.SILVER.DIM_SKIDDING_AND_OVERTURNING",
    "SEX_OF_DRIVER":       "SPARQ_CHALLENGE.SILVER.DIM_SEX_OF_DRIVER",
    "CASUALTY_SEVERITY":   "SPARQ_CHALLENGE.SILVER.DIM_CASUALTY_SEVERITY",
    "CASUALTY_CLASS":      "SPARQ_CHALLENGE.SILVER.DIM_CASUALTY_CLASS",
    "SEX_OF_CASUALTY":     "SPARQ_CHALLENGE.SILVER.DIM_SEX_OF_CASUALTY",
    "CASUALTY_TYPE":       "SPARQ_CHALLENGE.SILVER.DIM_CASUALTY_TYPE",
}
 
# Build a label lookup dict per column: {code (str) -> label}
dim_labels = {}
for col, table_path in dim_sources.items():
    dim_df = session.table(table_path).to_pandas()
    dim_df.columns = dim_df.columns.str.strip().str.upper()
    dim_labels[col] = dict(zip(dim_df["CODE"].astype(str), dim_df["LABEL"]))
 
print("Fact table row counts:")
for name, df in all_fact_dfs.items():
    print(f"  {name:<40}: {len(df):,}")
 
print(f"\nDimension tables loaded: {len(dim_labels)}")

In [ ]:
# ---------------------------------------------------------------------------
# Cell 3 — Quick preview of all tables
# ---------------------------------------------------------------------------
for table_name, df in all_fact_dfs.items():
    print(f"\n{'=' * 60}")
    print(f"  {table_name}")
    print(f"{'=' * 60}")
    print(df.head(3).to_string())
    print("\nColumn dtypes:")
    print(df.dtypes.to_string())

##### Phase 2: Univariate quality

In [ ]:
# ---------------------------------------------------------------------------
# Cell 4 — Null / empty / sentinel value analysis (all tables)
# ---------------------------------------------------------------------------
# Common sentinel values used in UK road safety data to represent unknowns
SENTINEL_VALUES = {"", "-1", "NULL", "null"}
 
 
def null_report(df: pd.DataFrame, table_name: str) -> pd.DataFrame:
    """
    Return a DataFrame with missing value counts and percentages per column.
 
    Treats NaN, empty strings, and common sentinel values (-1, NULL)
    as missing data, since all columns were loaded as TEXT from the stage.
 
    Parameters
    ----------
    df : pd.DataFrame
        Table to analyze.
    table_name : str
        Display name used in the printed header.
 
    Returns
    -------
    pd.DataFrame
        Sorted report with missing_count and missing_pct columns.
        Only columns with at least one missing value are included.
    """
    total_rows = len(df)
 
    missing_counts = df.isnull().sum()
    for sentinel in SENTINEL_VALUES:
        missing_counts += (df == sentinel).sum()
 
    report = pd.DataFrame({
        "missing_count": missing_counts,
        "missing_pct":   (missing_counts / total_rows * 100).round(2),
    }).sort_values("missing_pct", ascending=False)
 
    report = report[report["missing_count"] > 0]
 
    print(f"\n{'=' * 50}")
    print(f"  {table_name} — {total_rows:,} rows")
    print(f"{'=' * 50}")
    print(report.to_string())
 
    return report
 
 
null_accidents      = null_report(accidents,      "ACCIDENTS")
null_accidents_2019 = null_report(accidents_2019, "ACCIDENTS_2019_MIDYEAR_PROVISIONAL")
null_casualties     = null_report(casualties,     "CASUALTIES")
null_vehicles       = null_report(vehicles,       "VEHICLES")
null_blood_alcohol  = null_report(blood_alcohol,  "BLOOD_ALCOHOL_CONTENT")

In [ ]:
# ---------------------------------------------------------------------------
# Cell 5 — Null heatmap (horizontal bar chart per table)
# ---------------------------------------------------------------------------
null_reports = {
    "ACCIDENTS":      null_accidents,
    "ACCIDENTS_2019": null_accidents_2019,
    "CASUALTIES":     null_casualties,
    "VEHICLES":       null_vehicles,
    "BLOOD_ALCOHOL":  null_blood_alcohol,
}
 
fig, axes = plt.subplots(2, 3, figsize=(24, 14))
axes = axes.flatten()
 
for idx, (table_name, null_df) in enumerate(null_reports.items()):
    ax = axes[idx]
 
    if null_df.empty:
        ax.set_title(f"{table_name}\nNo missing values")
        ax.axis("off")
        continue
 
    plot_data = null_df.reset_index().rename(columns={"index": "column"})
    sns.barplot(
        data=plot_data,
        y="column",
        x="missing_pct",
        hue="column",
        legend=False,
        ax=ax,
        palette="Reds_r",
    )
    ax.set_title(f"{table_name}\n% Missing / Empty", fontweight="bold")
    ax.set_xlabel("% Missing")
    ax.set_ylabel("")
    ax.xaxis.set_major_formatter(mticker.PercentFormatter())
 
axes[-1].set_visible(False)
 
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Cell 6 — Type casting for key columns
# ---------------------------------------------------------------------------
 
# --- ACCIDENTS (2015–2018) ---
accidents["DATE"] = pd.to_datetime(accidents["DATE"], dayfirst=True, errors="coerce")
accidents["YEAR"] = accidents["DATE"].dt.year
accidents["MONTH"] = accidents["DATE"].dt.month
# Extract hour from TIME column (format HH:MM)
accidents["HOUR"] = pd.to_numeric(accidents["TIME"].str[:2], errors="coerce")
accidents["LONGITUDE"] = pd.to_numeric(accidents["LONGITUDE"], errors="coerce")
accidents["LATITUDE"]  = pd.to_numeric(accidents["LATITUDE"],  errors="coerce")
accidents["NUMBER_OF_VEHICLES"]   = pd.to_numeric(accidents["NUMBER_OF_VEHICLES"],   errors="coerce")
accidents["NUMBER_OF_CASUALTIES"] = pd.to_numeric(accidents["NUMBER_OF_CASUALTIES"], errors="coerce")
accidents["SPEED_LIMIT"] = pd.to_numeric(accidents["SPEED_LIMIT"], errors="coerce")
 
# --- ACCIDENTS_2019 — fewer columns available (no lat/lon, no LSOA) ---
accidents_2019["DATE"] = pd.to_datetime(accidents_2019["DATE"], dayfirst=True, errors="coerce")
accidents_2019["YEAR"]  = accidents_2019["DATE"].dt.year
accidents_2019["MONTH"] = accidents_2019["DATE"].dt.month
accidents_2019["HOUR"]  = pd.to_numeric(accidents_2019["TIME"].str[:2], errors="coerce")
accidents_2019["NUMBER_OF_VEHICLES"]   = pd.to_numeric(accidents_2019["NUMBER_OF_VEHICLES"],   errors="coerce")
accidents_2019["NUMBER_OF_CASUALTIES"] = pd.to_numeric(accidents_2019["NUMBER_OF_CASUALTIES"], errors="coerce")
accidents_2019["SPEED_LIMIT"] = pd.to_numeric(accidents_2019["SPEED_LIMIT"], errors="coerce")
 
# --- CASUALTIES ---
casualties["AGE_OF_CASUALTY"] = pd.to_numeric(casualties["AGE_OF_CASUALTY"], errors="coerce")
 
# --- VEHICLES ---
vehicles["AGE_OF_DRIVER"]  = pd.to_numeric(vehicles["AGE_OF_DRIVER"],  errors="coerce")
vehicles["AGE_OF_VEHICLE"] = pd.to_numeric(vehicles["AGE_OF_VEHICLE"], errors="coerce")
 
# --- BLOOD_ALCOHOL_CONTENT ---
blood_alcohol["BLOODALCOHOLLEVEL_MG_100ML"] = pd.to_numeric(
    blood_alcohol["BLOODALCOHOLLEVEL_MG_100ML"], errors="coerce"
)
 
print("Type casting complete.")

In [ ]:
# ---------------------------------------------------------------------------
# Cell 7 — Outlier detection (boxplots for all numeric columns)
# ---------------------------------------------------------------------------
numeric_columns = {
    "ACCIDENTS":      ["NUMBER_OF_VEHICLES", "NUMBER_OF_CASUALTIES", "SPEED_LIMIT"],
    "ACCIDENTS_2019": ["NUMBER_OF_VEHICLES", "NUMBER_OF_CASUALTIES", "SPEED_LIMIT"],
    "CASUALTIES":     ["AGE_OF_CASUALTY"],
    "VEHICLES":       ["AGE_OF_DRIVER", "AGE_OF_VEHICLE"],
    "BLOOD_ALCOHOL":  ["BLOODALCOHOLLEVEL_MG_100ML"],
}
 
table_map = {
    "ACCIDENTS":      accidents,
    "ACCIDENTS_2019": accidents_2019,
    "CASUALTIES":     casualties,
    "VEHICLES":       vehicles,
    "BLOOD_ALCOHOL":  blood_alcohol,
}
 
plot_pairs = [
    (table_name, col)
    for table_name, cols in numeric_columns.items()
    for col in cols
]
 
n_plots = len(plot_pairs)
n_cols  = 3
n_rows  = (n_plots + n_cols - 1) // n_cols
 
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 4))
axes = axes.flatten()
palette = sns.color_palette("muted")
 
for idx, (table_name, col) in enumerate(plot_pairs):
    df = table_map[table_name]
    sns.boxplot(
        x=df[col].dropna(),
        ax=axes[idx],
        color=palette[idx % len(palette)],
    )
    axes[idx].set_title(f"{table_name}\n{col}", fontweight="bold")
    axes[idx].set_xlabel("")
 
for idx in range(n_plots, len(axes)):
    axes[idx].set_visible(False)
 
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Cell 8 — Geographic validation (coordinates within UK bounding box)
# UK bounding box (approximate):
#   Latitude:  49.8N – 60.9N
#   Longitude: -8.7W – 1.8E
# ---------------------------------------------------------------------------
UK_LAT_MIN, UK_LAT_MAX = 49.8, 60.9
UK_LON_MIN, UK_LON_MAX = -8.7,  1.8
 
has_coords = accidents[["ACCIDENT_INDEX", "LATITUDE", "LONGITUDE"]].dropna(
    subset=["LATITUDE", "LONGITUDE"]
)
 
total_with_coords = len(has_coords)
total_accidents   = len(accidents)
 
print("=== Geographic Validation ===\n")
print(f"Accidents with coordinates    : {total_with_coords:,} ({total_with_coords / total_accidents * 100:.2f}%)")
print(f"Accidents without coordinates : {total_accidents - total_with_coords:,} ({(total_accidents - total_with_coords) / total_accidents * 100:.2f}%)")
 
out_of_bounds = has_coords[
    (has_coords["LATITUDE"]  < UK_LAT_MIN) | (has_coords["LATITUDE"]  > UK_LAT_MAX) |
    (has_coords["LONGITUDE"] < UK_LON_MIN) | (has_coords["LONGITUDE"] > UK_LON_MAX)
]
 
print(f"\nCoordinates outside UK bounding box: {len(out_of_bounds):,} ({len(out_of_bounds) / total_with_coords * 100:.3f}%)")
 
if len(out_of_bounds) > 0:
    print("\nSample out-of-bounds records:")
    print(out_of_bounds.head(10).to_string(index=False))
 
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
 
sns.histplot(has_coords["LATITUDE"],  bins=50, ax=axes[0], color=palette[0], kde=True)
axes[0].axvline(UK_LAT_MIN, color="red", linestyle="--", linewidth=1, label=f"Min ({UK_LAT_MIN})")
axes[0].axvline(UK_LAT_MAX, color="red", linestyle="--", linewidth=1, label=f"Max ({UK_LAT_MAX})")
axes[0].set_title("Latitude Distribution", fontweight="bold")
axes[0].set_xlabel("Latitude")
axes[0].legend(fontsize=8)
 
sns.histplot(has_coords["LONGITUDE"], bins=50, ax=axes[1], color=palette[1], kde=True)
axes[1].axvline(UK_LON_MIN, color="red", linestyle="--", linewidth=1, label=f"Min ({UK_LON_MIN})")
axes[1].axvline(UK_LON_MAX, color="red", linestyle="--", linewidth=1, label=f"Max ({UK_LON_MAX})")
axes[1].set_title("Longitude Distribution", fontweight="bold")
axes[1].set_xlabel("Longitude")
axes[1].legend(fontsize=8)
 
plt.tight_layout()
plt.show()

##### Phase 3: Referential integrity

In [ ]:
# ---------------------------------------------------------------------------
# Cell 9 — Schema diff: ACCIDENTS vs ACCIDENTS_2019
# ---------------------------------------------------------------------------
cols_accidents = set(accidents.columns)
cols_2019      = set(accidents_2019.columns)
 
print("Columns in ACCIDENTS but NOT in ACCIDENTS_2019:")
for col in sorted(cols_accidents - cols_2019):
    print(f"  - {col}")
 
print("\nColumns in ACCIDENTS_2019 but NOT in ACCIDENTS:")
for col in sorted(cols_2019 - cols_accidents):
    print(f"  - {col}")
 
print(f"\nShared columns: {len(cols_accidents & cols_2019)}")

In [ ]:
# ---------------------------------------------------------------------------
# Cell 10 — Duplicate detection (all fact tables)
# ---------------------------------------------------------------------------
dup_accidents      = accidents.duplicated(subset=["ACCIDENT_INDEX"]).sum()
dup_accidents_2019 = accidents_2019.duplicated(subset=["ACCIDENT_INDEX"]).sum()
dup_casualties     = casualties.duplicated(
    subset=["ACCIDENT_INDEX", "VEHICLE_REFERENCE", "CASUALTY_REFERENCE"]
).sum()
dup_vehicles      = vehicles.duplicated(
    subset=["ACCIDENT_INDEX", "VEHICLE_REFERENCE"]
).sum()
# BLOOD_ALCOHOL has no natural PK — check for fully identical rows
dup_blood_alcohol = blood_alcohol.duplicated().sum()
 
duplicate_summary = pd.DataFrame({
    "table": [
        "ACCIDENTS", "ACCIDENTS_2019", "CASUALTIES", "VEHICLES", "BLOOD_ALCOHOL",
    ],
    "primary_key": [
        "ACCIDENT_INDEX",
        "ACCIDENT_INDEX",
        "ACCIDENT_INDEX + VEHICLE_REF + CASUALTY_REF",
        "ACCIDENT_INDEX + VEHICLE_REF",
        "All columns (no natural PK)",
    ],
    "duplicates": [
        dup_accidents, dup_accidents_2019, dup_casualties, dup_vehicles, dup_blood_alcohol,
    ],
    "total_rows": [
        len(accidents), len(accidents_2019), len(casualties), len(vehicles), len(blood_alcohol),
    ],
})
duplicate_summary["duplicate_pct"] = (
    duplicate_summary["duplicates"] / duplicate_summary["total_rows"] * 100
).round(3)
 
print(duplicate_summary.to_string(index=False))

In [ ]:
# ---------------------------------------------------------------------------
# Cell 11 — Orphan check: accidents → casualties / vehicles
# ---------------------------------------------------------------------------
base_accident_indexes = set(accidents["ACCIDENT_INDEX"])
all_accident_indexes  = base_accident_indexes | set(accidents_2019["ACCIDENT_INDEX"])
 
orphan_casualties_base     = casualties[~casualties["ACCIDENT_INDEX"].isin(base_accident_indexes)]
orphan_casualties_combined = casualties[~casualties["ACCIDENT_INDEX"].isin(all_accident_indexes)]
orphan_vehicles_base       = vehicles[~vehicles["ACCIDENT_INDEX"].isin(base_accident_indexes)]
orphan_vehicles_combined   = vehicles[~vehicles["ACCIDENT_INDEX"].isin(all_accident_indexes)]
 
print("=== Orphan check vs ACCIDENTS (2015–2018) only ===")
print(f"  Casualties without parent: {len(orphan_casualties_base):,} ({len(orphan_casualties_base) / len(casualties) * 100:.2f}%)")
print(f"  Vehicles  without parent: {len(orphan_vehicles_base):,} ({len(orphan_vehicles_base) / len(vehicles) * 100:.2f}%)")
 
print("\n=== Orphan check vs ACCIDENTS + ACCIDENTS_2019 (combined) ===")
print(f"  Casualties without parent: {len(orphan_casualties_combined):,} ({len(orphan_casualties_combined) / len(casualties) * 100:.2f}%)")
print(f"  Vehicles  without parent: {len(orphan_vehicles_combined):,} ({len(orphan_vehicles_combined) / len(vehicles) * 100:.2f}%)")
 
if len(orphan_casualties_combined) > 0:
    print("\nSample orphan Accident_Index values (casualties):")
    print(orphan_casualties_combined["ACCIDENT_INDEX"].value_counts().head(10).to_string())

In [ ]:
# ---------------------------------------------------------------------------
# Cell 12 — Orphan check: casualties → vehicles
# Verifies that every (Accident_Index, Vehicle_Reference) pair in CASUALTIES
# exists in VEHICLES. Pedestrians may legitimately have no vehicle — flagged
# via Casualty_Class breakdown below.
# ---------------------------------------------------------------------------
valid_vehicle_pairs = set(zip(vehicles["ACCIDENT_INDEX"], vehicles["VEHICLE_REFERENCE"]))
casualty_pairs      = list(zip(casualties["ACCIDENT_INDEX"], casualties["VEHICLE_REFERENCE"]))
orphan_mask         = [pair not in valid_vehicle_pairs for pair in casualty_pairs]
 
orphan_casualty_vehicle = casualties[orphan_mask]
orphan_count = len(orphan_casualty_vehicle)
 
print("=== Orphan Casualty -> Vehicle Check ===\n")
print(
    f"Casualties with no matching (Accident_Index, Vehicle_Reference) in VEHICLES: "
    f"{orphan_count:,} ({orphan_count / len(casualties) * 100:.2f}%)"
)
 
if orphan_count > 0:
    print("\nSample orphan records:")
    print(
        orphan_casualty_vehicle[
            ["ACCIDENT_INDEX", "VEHICLE_REFERENCE", "CASUALTY_REFERENCE", "CASUALTY_CLASS"]
        ]
        .head(10)
        .to_string(index=False)
    )
    print("\nOrphan breakdown by Casualty Class (pedestrians expected here):")
    print(orphan_casualty_vehicle["CASUALTY_CLASS"].value_counts().to_string())

In [ ]:
# ---------------------------------------------------------------------------
# Cell 13 — Cardinality consistency check (STATS19 domain rule)
# Compares Number_of_Casualties / Number_of_Vehicles declared in ACCIDENTS
# against actual child row counts in CASUALTIES and VEHICLES tables.
# ---------------------------------------------------------------------------
actual_casualties = (
    casualties.groupby("ACCIDENT_INDEX")
    .size()
    .reset_index(name="actual_casualties")
)
actual_vehicles = (
    vehicles.groupby("ACCIDENT_INDEX")
    .size()
    .reset_index(name="actual_vehicles")
)
 
consistency_df = (
    accidents[["ACCIDENT_INDEX", "NUMBER_OF_CASUALTIES", "NUMBER_OF_VEHICLES"]]
    .merge(actual_casualties, on="ACCIDENT_INDEX", how="left")
    .merge(actual_vehicles,   on="ACCIDENT_INDEX", how="left")
)
 
consistency_df["casualty_mismatch"] = (
    consistency_df["NUMBER_OF_CASUALTIES"] != consistency_df["actual_casualties"]
)
consistency_df["vehicle_mismatch"] = (
    consistency_df["NUMBER_OF_VEHICLES"] != consistency_df["actual_vehicles"]
)
 
casualty_mismatch_count = consistency_df["casualty_mismatch"].sum()
vehicle_mismatch_count  = consistency_df["vehicle_mismatch"].sum()
total                   = len(consistency_df)
 
print("=== Cardinality Consistency Check (STATS19) ===\n")
print(f"Declared casualties != actual rows in CASUALTIES: {casualty_mismatch_count:,} ({casualty_mismatch_count / total * 100:.2f}%)")
print(f"Declared vehicles   != actual rows in VEHICLES:  {vehicle_mismatch_count:,} ({vehicle_mismatch_count / total * 100:.2f}%)")
 
consistency_df["casualty_diff"] = consistency_df["actual_casualties"] - consistency_df["NUMBER_OF_CASUALTIES"]
consistency_df["vehicle_diff"]  = consistency_df["actual_vehicles"]   - consistency_df["NUMBER_OF_VEHICLES"]
 
print("\nCasualty difference distribution (actual - declared):")
print(consistency_df["casualty_diff"].value_counts().sort_index().to_string())
 
print("\nVehicle difference distribution (actual - declared):")
print(consistency_df["vehicle_diff"].value_counts().sort_index().to_string())
 
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
 
for ax, col, title in zip(axes, ["casualty_diff", "vehicle_diff"], [
    "Casualty Count Difference (actual - declared)",
    "Vehicle Count Difference (actual - declared)",
]):
    diff_counts = consistency_df[col].value_counts().sort_index()
    sns.barplot(
        x=diff_counts.index.astype(str),
        y=diff_counts.values,
        hue=diff_counts.index.astype(str),
        legend=False,
        ax=ax,
        palette="muted",
    )
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Difference")
    ax.set_ylabel("Number of Accidents")
 
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Cell 14 — Dimension code validation against fact tables
# Checks that every code used in facts exists in its corresponding dimension.
# Unmatched codes will produce silent NULL joins in the Gold layer.
# ---------------------------------------------------------------------------
validation_checks = [
    # (fact_column,                   dim_key,                       fact_df,    fact_name)
    ("ACCIDENT_SEVERITY",             "ACCIDENT_SEVERITY",            accidents,  "ACCIDENTS"),
    ("WEATHER_CONDITIONS",            "WEATHER_CONDITIONS",           accidents,  "ACCIDENTS"),
    ("LIGHT_CONDITIONS",              "LIGHT_CONDITIONS",             accidents,  "ACCIDENTS"),
    ("ROAD_TYPE",                     "ROAD_TYPE",                    accidents,  "ACCIDENTS"),
    ("SPEED_LIMIT",                   "SPEED_LIMIT",                  accidents,  "ACCIDENTS"),
    ("ROAD_SURFACE_CONDITIONS",       "ROAD_SURFACE_CONDITIONS",      accidents,  "ACCIDENTS"),
    ("JUNCTION_DETAIL",               "JUNCTION_DETAIL",              accidents,  "ACCIDENTS"),
    ("JUNCTION_CONTROL",              "JUNCTION_CONTROL",             accidents,  "ACCIDENTS"),
    ("CARRIAGEWAY_HAZARDS",           "CARRIAGEWAY_HAZARDS",          accidents,  "ACCIDENTS"),
    ("URBAN_OR_RURAL_AREA",           "URBAN_OR_RURAL_AREA",          accidents,  "ACCIDENTS"),
    ("VEHICLE_TYPE",                  "VEHICLE_TYPE",                 vehicles,   "VEHICLES"),
    ("VEHICLE_MANOEUVRE",             "VEHICLE_MANOEUVRE",            vehicles,   "VEHICLES"),
    ("SKIDDING_AND_OVERTURNING",      "SKIDDING_AND_OVERTURNING",     vehicles,   "VEHICLES"),
    ("SEX_OF_DRIVER",                 "SEX_OF_DRIVER",                vehicles,   "VEHICLES"),
    ("CASUALTY_SEVERITY",             "CASUALTY_SEVERITY",            casualties, "CASUALTIES"),
    ("CASUALTY_CLASS",                "CASUALTY_CLASS",               casualties, "CASUALTIES"),
    ("SEX_OF_CASUALTY",               "SEX_OF_CASUALTY",              casualties, "CASUALTIES"),
    ("CASUALTY_TYPE",                 "CASUALTY_TYPE",                casualties, "CASUALTIES"),
]
 
validation_results = []
 
for fact_col, dim_key, fact_df, fact_name in validation_checks:
    if fact_col not in fact_df.columns:
        validation_results.append({
            "fact_table":   fact_name,
            "fact_column":  fact_col,
            "invalid_codes": "COLUMN NOT FOUND",
            "invalid_count": None,
            "invalid_pct":   None,
        })
        continue
 
    valid_codes = set(dim_labels[dim_key].keys())
    fact_codes  = fact_df[fact_col].dropna().astype(str)
    invalid_mask = (
        ~fact_codes.isin(valid_codes)
        & (fact_codes != "")
        & (fact_codes != "-1")
    )
    invalid_count  = invalid_mask.sum()
    invalid_pct    = round(invalid_count / len(fact_df) * 100, 3)
    invalid_sample = fact_codes[invalid_mask].unique()[:5].tolist()
 
    validation_results.append({
        "fact_table":    fact_name,
        "fact_column":   fact_col,
        "invalid_codes": str(invalid_sample) if invalid_count > 0 else "All codes valid",
        "invalid_count": invalid_count,
        "invalid_pct":   invalid_pct,
    })
 
validation_df = pd.DataFrame(validation_results).sort_values("invalid_count", ascending=False)
 
print("Dimension code validation results:")
print(validation_df.to_string(index=False))
 
issues = validation_df[
    validation_df["invalid_count"].notna() & (validation_df["invalid_count"] > 0)
]
if issues.empty:
    print("\nAll dimension codes validated successfully.")
else:
    print(f"\n{len(issues)} column(s) with unmatched codes — review before Gold layer build.")

##### Phase 4: Exploratory / business analysis

In [ ]:
# ---------------------------------------------------------------------------
# Cell 15 — Temporal distribution + year-over-year trends
# ---------------------------------------------------------------------------
 
# Apply severity label to both accident tables for YoY fatal count
severity_map = dim_labels["ACCIDENT_SEVERITY"]
accidents["SEVERITY_LABEL"]      = accidents["ACCIDENT_SEVERITY"].map(
    lambda x: severity_map.get(str(x), x)
)
accidents_2019["SEVERITY_LABEL"] = accidents_2019["ACCIDENT_SEVERITY"].map(
    lambda x: severity_map.get(str(x), x)
)
 
 
def yearly_metrics(df: pd.DataFrame, label: str) -> pd.DataFrame:
    """
    Compute yearly accident count, total casualties, and fatal accidents.
 
    Parameters
    ----------
    df : pd.DataFrame
        Accidents table with YEAR, NUMBER_OF_CASUALTIES, SEVERITY_LABEL columns.
    label : str
        Source label appended to the result for traceability.
 
    Returns
    -------
    pd.DataFrame
        One row per year with aggregated metrics.
    """
    agg = df.groupby("YEAR").agg(
        total_accidents=("ACCIDENT_INDEX",    "count"),
        total_casualties=("NUMBER_OF_CASUALTIES", "sum"),
        fatal_accidents=("SEVERITY_LABEL",    lambda x: (x == "Fatal").sum()),
    ).reset_index()
    agg["source"] = label
    return agg
 
 
metrics_combined = pd.concat([
    yearly_metrics(accidents,      "2015-2018"),
    yearly_metrics(accidents_2019, "2019 provisional"),
], ignore_index=True).sort_values("YEAR")
 
print("Year-over-year metrics:")
print(metrics_combined.to_string(index=False))
 
# --- Row 1: raw temporal distributions ---
fig, axes = plt.subplots(2, 3, figsize=(22, 12))
palette   = sns.color_palette("muted")
 
for row_idx, (table_name, df) in enumerate([
    ("ACCIDENTS (2015-2018)", accidents),
    ("ACCIDENTS_2019 (Provisional)", accidents_2019),
]):
    df["YEAR"].value_counts().sort_index().plot(
        kind="bar", ax=axes[row_idx][0], color=palette[0], edgecolor="white"
    )
    axes[row_idx][0].set_title(f"{table_name}\nBy Year", fontweight="bold")
    axes[row_idx][0].set_xlabel("Year")
    axes[row_idx][0].set_ylabel("Count")
 
    df["MONTH"].value_counts().sort_index().plot(
        kind="bar", ax=axes[row_idx][1], color=palette[1], edgecolor="white"
    )
    axes[row_idx][1].set_title(f"{table_name}\nBy Month", fontweight="bold")
    axes[row_idx][1].set_xlabel("Month")
    axes[row_idx][1].set_ylabel("")
 
    df["HOUR"].dropna().value_counts().sort_index().plot(
        kind="bar", ax=axes[row_idx][2], color=palette[2], edgecolor="white"
    )
    axes[row_idx][2].set_title(f"{table_name}\nBy Hour of Day", fontweight="bold")
    axes[row_idx][2].set_xlabel("Hour")
    axes[row_idx][2].set_ylabel("")
 
plt.tight_layout()
plt.show()
 
# --- Row 2: YoY trend lines ---
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
 
for ax, (metric, title) in zip(axes, [
    ("total_accidents",   "Total Accidents per Year"),
    ("total_casualties",  "Total Casualties per Year"),
    ("fatal_accidents",   "Fatal Accidents per Year"),
]):
    ax.plot(
        metrics_combined["YEAR"],
        metrics_combined[metric],
        marker="o",
        linewidth=2,
        color=palette[0],
    )
    provisional = metrics_combined[metrics_combined["source"] == "2019 provisional"]
    if not provisional.empty:
        ax.annotate(
            "Provisional",
            xy=(provisional["YEAR"].values[0], provisional[metric].values[0]),
            xytext=(10, -15),
            textcoords="offset points",
            fontsize=8,
            color="grey",
        )
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Year")
    ax.set_ylabel("Count")
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
 
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Cell 16 — Blood alcohol distribution
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
 
sns.histplot(
    blood_alcohol["BLOODALCOHOLLEVEL_MG_100ML"].dropna(),
    bins=30,
    ax=axes[0],
    color=palette[3],
    kde=True,
)
axes[0].set_title("Blood Alcohol Level (mg/100ml) Distribution", fontweight="bold")
axes[0].set_xlabel("mg / 100ml")
axes[0].set_ylabel("Count")
 
sns.boxplot(
    data=blood_alcohol,
    x="SEVERITYOFINJURY",
    y="BLOODALCOHOLLEVEL_MG_100ML",
    hue="SEVERITYOFINJURY",
    legend=False,
    ax=axes[1],
    palette="muted",
)
axes[1].set_title("Blood Alcohol Level by Injury Severity", fontweight="bold")
axes[1].set_xlabel("Severity of Injury")
axes[1].set_ylabel("Blood Alcohol Level (mg/100ml)")
 
plt.tight_layout()
plt.show()
 
print("\nBlood Alcohol — Casualty Class breakdown:")
print(blood_alcohol["CASUALTYCLASS"].value_counts().to_string())
 
print("\nBlood Alcohol — Age Band breakdown:")
print(blood_alcohol["AGEBAND"].value_counts().to_string())
 
print("\nBlood Alcohol — Sex of Casualty breakdown:")
print(blood_alcohol["SEXOFCASUALTY"].value_counts().to_string())

In [ ]:
# ---------------------------------------------------------------------------
# Cell 17 — Severity distribution
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
 
severity_counts = accidents["SEVERITY_LABEL"].value_counts()
colors = ["#d62728", "#ff7f0e", "#2ca02c"]  # Fatal, Serious, Slight
 
severity_counts.plot(
    kind="bar",
    ax=axes[0],
    color=colors,
    edgecolor="white",
)
axes[0].set_title("Accident Count by Severity (2015-2018)", fontweight="bold")
axes[0].set_xlabel("Severity")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=0)
 
axes[1].pie(
    severity_counts,
    labels=severity_counts.index,
    autopct="%1.1f%%",
    colors=colors,
    startangle=90,
    wedgeprops={"edgecolor": "white", "linewidth": 1.5},
)
axes[1].set_title("Severity Share (2015-2018)", fontweight="bold")
 
plt.tight_layout()
plt.show()
 
print("\nSeverity breakdown:")
print(
    severity_counts.to_frame("count")
    .assign(pct=lambda x: (x["count"] / x["count"].sum() * 100).round(2))
    .to_string()
)

In [ ]:
# ---------------------------------------------------------------------------
# Cell 18 — Cross-tabulation: Severity x contextual factors
# Row-normalized percentages so groups of different sizes are comparable.
# ---------------------------------------------------------------------------
 
# Apply dimension labels to a working copy
acc_labeled = accidents.copy()
for col in ["SPEED_LIMIT", "URBAN_OR_RURAL_AREA", "LIGHT_CONDITIONS"]:
    acc_labeled[col + "_LABEL"] = acc_labeled[col].map(
        lambda x, m=dim_labels[col]: m.get(str(x), x)
    )
 
fig, axes = plt.subplots(1, 3, figsize=(22, 7))
 
for ax, (col_label, x_label) in zip(axes, [
    ("SPEED_LIMIT_LABEL",         "Speed Limit (mph)"),
    ("URBAN_OR_RURAL_AREA_LABEL", "Area Type"),
    ("LIGHT_CONDITIONS_LABEL",    "Light Conditions"),
]):
    crosstab = pd.crosstab(
        acc_labeled[col_label],
        acc_labeled["SEVERITY_LABEL"],
        normalize="index",
    ) * 100
 
    crosstab.plot(
        kind="bar",
        ax=ax,
        colormap="RdYlGn_r",
        edgecolor="white",
        width=0.75,
    )
    ax.set_title(f"Severity by {x_label}\n(% within group)", fontweight="bold")
    ax.set_xlabel(x_label)
    ax.set_ylabel("% of Accidents")
    ax.tick_params(axis="x", rotation=45)
    ax.legend(title="Severity", fontsize=8)
 
plt.tight_layout()
plt.show()
 
print("\nSeverity x Speed Limit (row %):")
print(
    pd.crosstab(
        acc_labeled["SPEED_LIMIT_LABEL"],
        acc_labeled["SEVERITY_LABEL"],
        normalize="index",
    ).mul(100).round(2).to_string()
)
 
print("\nSeverity x Urban/Rural (row %):")
print(
    pd.crosstab(
        acc_labeled["URBAN_OR_RURAL_AREA_LABEL"],
        acc_labeled["SEVERITY_LABEL"],
        normalize="index",
    ).mul(100).round(2).to_string()
)
 
print("\nSeverity x Light Conditions (row %):")
print(
    pd.crosstab(
        acc_labeled["LIGHT_CONDITIONS_LABEL"],
        acc_labeled["SEVERITY_LABEL"],
        normalize="index",
    ).mul(100).round(2).to_string()
)

In [ ]:
# ---------------------------------------------------------------------------
# Cell 19 — Categorical column distributions
# ---------------------------------------------------------------------------
categorical_cols = [
    "WEATHER_CONDITIONS",
    "LIGHT_CONDITIONS",
    "ROAD_TYPE",
    "SPEED_LIMIT",
    "ROAD_SURFACE_CONDITIONS",
]
 
fig, axes = plt.subplots(2, 3, figsize=(24, 14))
axes = axes.flatten()
 
for idx, col in enumerate(categorical_cols):
    ax = axes[idx]
 
    mapped  = accidents[col].map(lambda x, m=dim_labels[col]: m.get(str(x), x))
    counts  = mapped.value_counts().head(10)
 
    sns.barplot(
        x=counts.values,
        y=counts.index,
        hue=counts.index,
        legend=False,
        ax=ax,
        palette="muted",
    )
    ax.set_title(f"Top values — {col}", fontweight="bold")
    ax.set_xlabel("Count")
    ax.set_ylabel("")
 
    # Print unexpected codes not found in dimension table
    unmapped = accidents[col][
        ~accidents[col].astype(str).isin(dim_labels[col].keys())
    ]
    unmapped_unique = unmapped[unmapped.notna() & (unmapped != "")].unique()
    if len(unmapped_unique) > 0:
        print(f"\nUnexpected codes in {col} (not in dimension table):")
        print(unmapped_unique)
 
axes[-1].set_visible(False)
 
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Cell 20 — Correlation heatmap (numeric variables)
# Aggregates vehicles and casualties to accident level before joining
# to avoid row-count inflation from the one-to-many relationships.
# ---------------------------------------------------------------------------
vehicles_agg = vehicles.groupby("ACCIDENT_INDEX").agg(
    avg_driver_age=("AGE_OF_DRIVER",  "mean"),
    avg_vehicle_age=("AGE_OF_VEHICLE", "mean"),
).reset_index()
 
casualties_agg = casualties.groupby("ACCIDENT_INDEX").agg(
    avg_casualty_age=("AGE_OF_CASUALTY", "mean"),
).reset_index()
 
numeric_df = (
    accidents[[
        "ACCIDENT_INDEX",
        "NUMBER_OF_VEHICLES",
        "NUMBER_OF_CASUALTIES",
        "SPEED_LIMIT",
        "HOUR",
        "MONTH",
    ]]
    .merge(vehicles_agg,   on="ACCIDENT_INDEX", how="left")
    .merge(casualties_agg, on="ACCIDENT_INDEX", how="left")
    .drop(columns=["ACCIDENT_INDEX"])
)
 
corr_matrix = numeric_df.corr(numeric_only=True)
 
fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    linewidths=0.5,
    ax=ax,
    annot_kws={"size": 10},
)
ax.set_title(
    "Correlation Matrix — Accident-level Numeric Variables",
    fontweight="bold",
    pad=15,
)
plt.tight_layout()
plt.show()
 
print("\nStrongest correlations (absolute value > 0.1):")
corr_pairs = (
    corr_matrix
    .unstack()
    .reset_index()
    .rename(columns={"level_0": "var_1", "level_1": "var_2", 0: "correlation"})
)
corr_pairs = corr_pairs[corr_pairs["var_1"] < corr_pairs["var_2"]]
corr_pairs = corr_pairs[corr_pairs["correlation"].abs() > 0.1]
corr_pairs = corr_pairs.sort_values("correlation", ascending=False)
print(corr_pairs.to_string(index=False))

##### Phase 5: Summary

In [ ]:
# ---------------------------------------------------------------------------
# Cell 21 — EDA summary report (all tables)
# ---------------------------------------------------------------------------
table_pk_map = [
    ("ACCIDENTS",      accidents,      ["ACCIDENT_INDEX"]),
    ("ACCIDENTS_2019", accidents_2019, ["ACCIDENT_INDEX"]),
    ("CASUALTIES",     casualties,     ["ACCIDENT_INDEX", "VEHICLE_REFERENCE", "CASUALTY_REFERENCE"]),
    ("VEHICLES",       vehicles,       ["ACCIDENT_INDEX", "VEHICLE_REFERENCE"]),
    ("BLOOD_ALCOHOL",  blood_alcohol,  None),
]
 
summary_rows = []
 
for table_name, df, primary_key in table_pk_map:
    total_rows = len(df)
    duplicates = df.duplicated(subset=primary_key).sum() if primary_key else df.duplicated().sum()
    missing    = df.isnull().sum() + (df == "").sum() + (df == "-1").sum()
    max_missing_pct = round(missing.max() / total_rows * 100, 2)
 
    summary_rows.append({
        "table":         table_name,
        "rows":          f"{total_rows:,}",
        "columns":       df.shape[1],
        "duplicates":    duplicates,
        "max_null_pct":  f"{max_missing_pct}%",
        "needs_casting": "Yes — all columns loaded as TEXT",
    })
 
print("EDA Summary:")
print(pd.DataFrame(summary_rows).to_string(index=False))
print("\nEDA complete — ready for Silver / Gold layer modeling.")